In [2]:
import cv2
import numpy as np

def detect_horizon_via_sky_segmentation(frame):
    # Resize frame so its easy to see
    frame = cv2.resize(frame, (1280, 720))

    # Convert to HSV since I will try to segment using color gradients
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    # Define HSV range for blue sky, needs to cover light and dark blue. 
    lower_blue_sky = np.array([90, 20, 60])
    upper_blue_sky = np.array([135, 255, 255])

    # Define HSV range for white/light-gray clouds (found via trial and error)
    lower_cloud = np.array([0, 0, 100])
    upper_cloud = np.array([135, 60, 255])

    # Create sky mask (sky + clouds)
    blue_sky_mask = cv2.inRange(hsv, lower_blue_sky, upper_blue_sky)
    cloud_mask = cv2.inRange(hsv, lower_cloud, upper_cloud)
    sky_mask = cv2.bitwise_or(blue_sky_mask, cloud_mask)

    # Invert to get ground mask
    ground_mask = cv2.bitwise_not(sky_mask)

    # Initialize array to store horizon line (y-coordinate) per column
    height, width = sky_mask.shape
    horizon_line = np.zeros(width, dtype=np.int32)

    # Figure out the top-most ground pixel in each one dimesional column
    for x in range(width):
        column = ground_mask[:, x]
        nonzero = np.where(column > 0)[0]
        if len(nonzero) > 0:
            horizon_line[x] = nonzero[0]  # First ground pixel, scanned from top to bottom
        else:
            horizon_line[x] = height  # All sky in this column

    # Draw horizon line in forest green: BGR = (34, 139, 34)
    # Smoothing outliers in the horizon line
    max_allowed_jump = 25 # pixels

    for x in range(1, width):
        prev = horizon_line[x - 1]
        curr = horizon_line[x]
        
        # If jump is too big, override the current point
        if abs(curr - prev) > max_allowed_jump:
            horizon_line[x] = prev  # or use avg: (prev + curr) // 2

        # Draw the line
        cv2.line(
            frame,
            (x - 1, horizon_line[x - 1]),
            (x, horizon_line[x]),
            (0, 0, 255), 
            2
        )


    return frame

#video loop to keep reading video until it ends
cap = cv2.VideoCapture('video.mp4')

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    result = detect_horizon_via_sky_segmentation(frame)
    cv2.imshow('Horizon via Sky Segmentation', result)

    if cv2.waitKey(25) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


KeyboardInterrupt: 

Hello world
